In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/shubhangig870@gmail.com/sephoraa/1_Setup/Utility

In [0]:
dbutils.widgets.text("catalog","sephoraa","catalog")
dbutils.widgets.text("data_source","brands","data_source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")


#### Silver procecing 

In [0]:
df_bronze = spark.sql(f"select * from {catalog}.{bronze_schema}.{data_source};")
display(df_bronze)

In [0]:
df_bronze.printSchema()

In [0]:
df_bronze.columns

In [0]:
#Drop Duplicates
df_silver = df_bronze.dropDuplicates()

In [0]:
# from REMOVE the extra speces in the column

df_silver=df_silver.withColumn("brand_id",F.trim(F.col("brand_id"))
).withColumn("brand_name",F.trim(F.col("brand_name"))
).withColumn("country_of_origin",F.trim(F.col("country_of_origin"))
).withColumn("founded_year",F.trim(F.col("founded_year"))
).withColumn("is_luxury",F.trim(F.col("is_luxury"))
).withColumn("created_at",F.trim(F.col("created_at"))
).withColumn("brand_id",F.trim(F.col("brand_id"))
).withColumn("updated_at",F.trim(F.col("updated_at"))
).withColumn("ingestion_date",F.trim(F.col("ingestion_date"))
)



In [0]:
# Null record count
from pyspark.sql.functions import col, count, when
null_count=df_silver.select([count(when(col(c).isNull(),c)).alias(c)for c in df_silver.columns])
null_count.display()

#### Cleanning data in table

In [0]:
# brand_id

new=df_silver.filter(col("brand_id").rlike("^//BRD"))
display(new)

In [0]:
#  brand_name
# replece
df_silver = df_silver.withColumn("brand_name",when(col("brand_name").cast("string")=="#N/A","unknown").otherwise(col("brand_name")))

df_silver = df_silver.withColumn("brand_name",when(col("brand_name").isNull(),"unknown").otherwise(col("brand_name")))

dup = df_silver.groupBy("brand_name").count().filter(col("count")>1)
display(dup)

null_count = df_bronze.filter(col("brand_name").rlike("[-_=\\[\\(<\\>\\?#*~%$&@]"))
display(null_count)



In [0]:
#  country_of_origin
# Null record count
from pyspark.sql.functions import col, count, when
null_count=df_silver.select([count(when(col(c).isNull(),c)).alias(c)for c in df_silver.columns])
display(null_count)

In [0]:
display(df_silver)

In [0]:


from pyspark.sql.functions import col

df_silver = df_silver.withColumn("founded_year", col("founded_year").cast("int"))
display(df_silver)

In [0]:
#  is_luxury

from pyspark.sql.functions import col, upper

df_silver = df_silver.withColumn("is_luxury", upper(col("is_luxury")))
display(df_silver)

dup = df_silver.groupBy("is_luxury").count ().filter(col("count")>1)
display(dup)

df_silver = df_silver.withColumn("is_luxury",when(col("is_luxury")=="N/A","unknown").otherwise(col("is_luxury")))
display(df_silver)


In [0]:
#  created_at

from pyspark.sql.functions import col,when
from pyspark.sql import functions as F
df_silver = df_silver.withColumn(
    "created_at",
    F.coalesce(
        # Date-only formats
        F.try_to_date(F.trim(F.col("created_at")), F.lit("yyyy/MM/dd")),
        F.try_to_date(F.trim(F.col("created_at")), F.lit("dd/MM/yyyy")),
        F.try_to_date(F.trim(F.col("created_at")), F.lit("yyyy-MM-dd")),
        F.try_to_date(F.trim(F.col("created_at")), F.lit("dd-MM-yyyy")),
        # Timestamp formats
        F.try_to_timestamp(F.trim(F.col("created_at")), F.lit("yyyy-MM-dd HH:mm:ss")),
        F.try_to_timestamp(F.trim(F.col("created_at")), F.lit("yyyy/MM/dd HH:mm:ss"))
    )
)
df_silver = df_silver.withColumn("created_at", F.to_date("created_at"))
# display(df_silver)

from pyspark.sql.functions import col, when, current_date

df_silver = df_silver.withColumn(
    "created_at",
    when(
        col("created_at").isNull(),
        current_date()
    ).otherwise(col("created_at"))
)

display(df_silver)

In [0]:
#  updated_at
from pyspark.sql.functions import col,when
from pyspark.sql import functions as F
df_silver = df_silver.withColumn(
    "updated_at",
    F.coalesce(
        # Date-only formats
        F.try_to_date(F.trim(F.col("updated_at")), F.lit("yyyy/MM/dd")),
        F.try_to_date(F.trim(F.col("updated_at")), F.lit("dd/MM/yyyy")),
        F.try_to_date(F.trim(F.col("updated_at")), F.lit("yyyy-MM-dd")),
        F.try_to_date(F.trim(F.col("updated_at")), F.lit("dd-MM-yyyy")),
        # Timestamp formats
        F.try_to_timestamp(F.trim(F.col("updated_at")), F.lit("yyyy-MM-dd HH:mm:ss")),
        F.try_to_timestamp(F.trim(F.col("updated_at")), F.lit("yyyy/MM/dd HH:mm:ss"))
    )
)
df_silver = df_silver.withColumn("updated_at", F.to_date("updated_at"))
# display(df_silver)

from pyspark.sql.functions import col, when, current_date

df_silver = df_silver.withColumn(
    "updated_at",
    when(
        col("updated_at").isNull(),
        current_date()
    ).otherwise(col("updated_at"))
)

display(df_silver)

In [0]:


#  ingestion_date

from pyspark.sql.functions import col,when
from pyspark.sql import functions as F
df_silver = df_silver.withColumn(
    "ingestion_date",
    F.coalesce(
        # Date-only formats
        F.try_to_date(F.trim(F.col("ingestion_date")), F.lit("yyyy/MM/dd")),
        F.try_to_date(F.trim(F.col("ingestion_date")), F.lit("dd/MM/yyyy")),
        F.try_to_date(F.trim(F.col("ingestion_date")), F.lit("yyyy-MM-dd")),
        F.try_to_date(F.trim(F.col("ingestion_date")), F.lit("dd-MM-yyyy")),
        # Timestamp formats
        F.try_to_timestamp(F.trim(F.col("ingestion_date")), F.lit("yyyy-MM-dd HH:mm:ss")),
        F.try_to_timestamp(F.trim(F.col("ingestion_date")), F.lit("yyyy/MM/dd HH:mm:ss"))
    )
)
df_silver = df_silver.withColumn("ingestion_date", F.to_date("ingestion_date"))
# display(df_silver)

from pyspark.sql.functions import col, when, current_date

df_silver = df_silver.withColumn(
    "ingestion_date",
    when(
        col("ingestion_date").isNull(),
        current_date()
    ).otherwise(col("ingestion_date"))
)

display(df_silver)